# local-pdf-rag — retriever fine-tune (Colab / GPU)

Same code as `ragpdf/finetune_retriever.py` in the repo, run here for GPU instead of the local CPU box. Runtime > Change runtime type > **GPU** before running.

**Before running:** upload two small files from your local `local-pdf-rag` checkout when the upload cell asks for them:
- `eval/titles.json` (Set A, ~20 KB)
- `index/chunks.json` (~1.5 MB)

Neither is in the GitHub repo (both are gitignored build/derived artifacts), so this notebook takes them by upload instead of `git clone`.

At the end, run the last cell to download `finetuned-retriever.zip` and unzip it into `models/finetuned-retriever/` in your local checkout.

In [ ]:
!pip install -q sentence-transformers datasets accelerate
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none -- set Runtime > Change runtime type > GPU")

In [ ]:
from google.colab import files
print("Select eval/titles.json and index/chunks.json (both at once is fine):")
uploaded = files.upload()
assert "titles.json" in uploaded, "titles.json not uploaded"
assert "chunks.json" in uploaded, "chunks.json not uploaded"

In [ ]:
# Identical logic to ragpdf/finetune_retriever.py -- split_titles() and build_pairs().
# Kept in lockstep by hand: if you change one, change both, or the Colab-trained
# model and eval/compare_finetuned.py's held-out split will silently disagree.
import json

def split_titles(titles: list[dict], held_out_every: int = 5):
    train = [t for i, t in enumerate(titles) if i % held_out_every != 0]
    test = [t for i, t in enumerate(titles) if i % held_out_every == 0]
    return train, test

def build_pairs(titles: list[dict], chunks: list[dict]) -> list[tuple[str, str]]:
    pairs, skipped = [], []
    for t in titles:
        gold = range(t["gold_pages"][0], t["gold_pages"][-1] + 1)
        match = next((c for c in chunks if c["page"] in gold), None)
        if match is None:
            skipped.append(t["query"])
            continue
        pairs.append((t["query"], match["text"]))
    if skipped:
        print(f"  {len(skipped)} title(s) with no matching chunk, skipped: {skipped}")
    return pairs

with open("titles.json", encoding="utf-8") as f:
    titles = json.load(f)
with open("chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)

train_titles, test_titles = split_titles(titles)
print(f"{len(train_titles)} train titles, {len(test_titles)} held out for eval (every 5th, by position)")
pairs = build_pairs(train_titles, chunks)
print(f"{len(pairs)} (title, chunk) training pairs")

In [ ]:
from sentence_transformers import InputExample, SentenceTransformer
from sentence_transformers.sentence_transformer import losses
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
BASE_MODEL = "all-mpnet-base-v2"
EPOCHS = 4
BATCH_SIZE = 8   # same as the local run; raise it if you want to use the GPU headroom

model = SentenceTransformer(BASE_MODEL, device=device)
examples = [InputExample(texts=[a, b]) for a, b in pairs]
loader = DataLoader(examples, shuffle=True, batch_size=BATCH_SIZE)
loss = losses.MultipleNegativesRankingLoss(model)

model.fit(
    train_objectives=[(loader, loss)],
    epochs=EPOCHS,
    warmup_steps=max(1, int(0.1 * len(loader) * EPOCHS)),
    show_progress_bar=True,
)

model.save("finetuned-retriever")
print("saved to ./finetuned-retriever")

In [ ]:
import shutil
shutil.make_archive("finetuned-retriever", "zip", "finetuned-retriever")
files.download("finetuned-retriever.zip")
print("Unzip this into models/finetuned-retriever/ in your local checkout, "
      "then run: python -m eval.compare_finetuned")